# Hospital Mortality Prediction - MIMIC-III ICU Data
## Computational Machine Learning - Final Project 2025

**Student:** Corneel Dils  
**Task:** Binary Classification - Predict `HOSPITAL_EXPIRE_FLAG`  
**Dataset:** MIMIC-III ICU Patient Data  

---

### Table of Contents
1. [Setup & Data Loading](#section1)
2. [Hospital History Features](#section2)
3. [ICD9 Diagnosis Features](#section3)
4. [Remove Leakage Columns](#section4)
5. [Age Calculation](#section5)
6. [Missing Value Imputation](#section6)
7. [Categorical Encoding](#section7)
8. [Medical Feature Engineering](#section8)
9. [Feature Scaling](#section9)
10. [Validation](#section10)
11. [Model Training - Gradient Boosting with Optuna](#section11)
12. [Generate Predictions](#section12)

---
<a id='section1'></a>
## 1. Setup & Data Loading

In [1]:
# =============================================================================
# IMPORTS
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import roc_auc_score
import optuna
from pathlib import Path
from datetime import datetime
import pickle
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("MIMIC-III HOSPITAL MORTALITY PREDICTION")
print("Computational Machine Learning - Final Project 2025")
print("="*80)

MIMIC-III HOSPITAL MORTALITY PREDICTION
Computational Machine Learning - Final Project 2025


c:\Code\BSEcode\Computational Machine Learning\CML_final_project_2025\Classification _HEF\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# =============================================================================
# LOAD DATA
# =============================================================================

print("\n" + "="*80)
print("SECTION 1: LOADING DATA")
print("="*80)

# Set data path - adjust as needed for your environment

# Load main datasets
train_raw = pd.read_csv('../data/mimic_train_HEF.csv')
test_raw = pd.read_csv('../data/mimic_test_HEF.csv')


# Load diagnoses
diagnoses_raw = pd.read_csv('../data/extra_data/MIMIC_diagnoses.csv')

print(f"\n✓ Loaded successfully:")
print(f"  Train: {train_raw.shape}")
print(f"  Test: {test_raw.shape}")
print(f"  Diagnoses: {diagnoses_raw.shape}")

# Create working copies
train = train_raw.copy()
test = test_raw.copy()
diagnoses = diagnoses_raw.copy()

# Standardize diagnoses column names to uppercase
diagnoses.columns = diagnoses.columns.str.upper()

# Display dataset overview
print(f"\n--- Dataset Overview ---")
print(f"Train samples: {len(train):,}")
print(f"Test samples: {len(test):,}")
print(f"Diagnosis records: {len(diagnoses):,}")

# Check for duplicates
print(f"\n--- Duplicate Check ---")
train_dupes = train['icustay_id'].duplicated().sum()
test_dupes = test['icustay_id'].duplicated().sum()
print(f"Train duplicates: {train_dupes}")
print(f"Test duplicates: {test_dupes}")

if train_dupes == 0 and test_dupes == 0:
    print("✓ No duplicates")

# ID structure
print(f"\n--- ID Structure ---")
print(f"Unique patients (subject_id): {train['subject_id'].nunique():,}")
print(f"Unique admissions (hadm_id): {train['hadm_id'].nunique():,}")
print(f"Unique ICU stays (icustay_id): {train['icustay_id'].nunique():,}")

# Patient visit statistics
visits_per_patient = train.groupby('subject_id').size()
print(f"\n--- Visit Statistics ---")
print(f"Mean ICU stays per patient: {visits_per_patient.mean():.2f}")
print(f"Patients with multiple visits: {(visits_per_patient > 1).sum():,} ({(visits_per_patient > 1).sum()/len(visits_per_patient)*100:.1f}%)")

print("\n" + "="*80)
print("✓ SECTION 1 COMPLETE")
print("="*80)


SECTION 1: LOADING DATA

✓ Loaded successfully:
  Train: (20885, 44)
  Test: (5221, 39)
  Diagnoses: (651047, 4)

--- Dataset Overview ---
Train samples: 20,885
Test samples: 5,221
Diagnosis records: 651,047

--- Duplicate Check ---
Train duplicates: 0
Test duplicates: 0
✓ No duplicates

--- ID Structure ---
Unique patients (subject_id): 16,317
Unique admissions (hadm_id): 19,749
Unique ICU stays (icustay_id): 20,885

--- Visit Statistics ---
Mean ICU stays per patient: 1.28
Patients with multiple visits: 2,940 (18.0%)

✓ SECTION 1 COMPLETE


---
<a id='section2'></a>
## 2. Hospital History Features

Creating features based on patient's previous hospital/ICU visits:
- `n_previous_icu_stays`: Count of prior ICU visits for this patient
- `is_first_icu_visit`: Binary flag for first-time ICU patients
- `is_frequent_flyer`: Binary flag for patients with 3+ total visits

In [6]:
# =============================================================================
# HOSPITAL HISTORY FEATURES
# =============================================================================

print("\n" + "="*80)
print("SECTION 2: CREATING HOSPITAL HISTORY FEATURES")
print("="*80)

def create_hospital_history_features(df, df_name="dataset"):
    """
    Create features based on patient's visit history.
    
    Features created:
    - n_previous_icu_stays: Number of previous ICU visits for this patient
    - is_first_icu_visit: Binary flag for first-time ICU patients
    - is_frequent_flyer: Binary flag for patients with 3+ visits
    """
    print(f"\n--- Processing {df_name} ---")
    
    df = df.copy()
    
    # Sort by patient and time
    if 'ADMITTIME' in df.columns:
        df['ADMITTIME'] = pd.to_datetime(df['ADMITTIME'], errors='coerce')
        df = df.sort_values(['subject_id', 'ADMITTIME'])
        print("  ✓ Sorted by patient and admission time")
    else:
        df = df.sort_values(['subject_id', 'hadm_id', 'icustay_id'])
        print("  ✓ Sorted by patient and IDs")
    
    # Feature 1: Previous ICU stays (cumulative count before current visit)
    df['n_previous_icu_stays'] = df.groupby('subject_id').cumcount()
    
    # Feature 2: First visit flag
    df['is_first_icu_visit'] = (df['n_previous_icu_stays'] == 0).astype(int)
    
    # Feature 3: Frequent flyer (3+ visits in entire dataset)
    total_visits = df.groupby('subject_id').size()
    frequent_patients = total_visits[total_visits >= 3].index
    df['is_frequent_flyer'] = df['subject_id'].isin(frequent_patients).astype(int)
    
    # Validation
    print(f"\n  Validation:")
    print(f"    n_previous_icu_stays - Min: {df['n_previous_icu_stays'].min()}, Max: {df['n_previous_icu_stays'].max()}, Mean: {df['n_previous_icu_stays'].mean():.2f}")
    print(f"    is_first_icu_visit - First visits: {df['is_first_icu_visit'].sum()} ({df['is_first_icu_visit'].mean()*100:.1f}%)")
    print(f"    is_frequent_flyer - Frequent flyers: {df['is_frequent_flyer'].sum()} ({df['is_frequent_flyer'].mean()*100:.1f}%)")
    
    # Check: Every patient's first row should have n_previous = 0
    first_rows = df.groupby('subject_id').first()
    assert (first_rows['n_previous_icu_stays'] == 0).all(), "ERROR: Not all first visits have n_previous = 0!"
    print(f"    ✓ Check passed: All first visits correctly marked")
    
    return df

# Apply to train and test
train = create_hospital_history_features(train, "train")
test = create_hospital_history_features(test, "test")

# Save test IDs NOW (after sorting!) - CRITICAL for submission alignment
test_ids = test['icustay_id'].copy()
print(f"\n✓ Saved {len(test_ids)} test IDs in correct order")

print("\n" + "="*80)
print("✓ SECTION 2 COMPLETE - Hospital history features created")
print("="*80)


SECTION 2: CREATING HOSPITAL HISTORY FEATURES

--- Processing train ---
  ✓ Sorted by patient and admission time

  Validation:
    n_previous_icu_stays - Min: 0, Max: 24, Mean: 0.40
    is_first_icu_visit - First visits: 16317 (78.1%)
    is_frequent_flyer - Frequent flyers: 3388 (16.2%)
    ✓ Check passed: All first visits correctly marked

--- Processing test ---
  ✓ Sorted by patient and admission time

  Validation:
    n_previous_icu_stays - Min: 0, Max: 4, Mean: 0.09
    is_first_icu_visit - First visits: 4847 (92.8%)
    is_frequent_flyer - Frequent flyers: 172 (3.3%)
    ✓ Check passed: All first visits correctly marked

✓ Saved 5221 test IDs in correct order

✓ SECTION 2 COMPLETE - Hospital history features created


---
<a id='section3'></a>
## 3. ICD9 Diagnosis Features

Extracting features from the external ICD-9 diagnosis codes:
- Number of diagnoses per admission
- Primary diagnosis category (3-digit ICD9)
- Major disease category mapping
- High-risk condition flags (sepsis, heart failure, respiratory failure, etc.)

In [7]:
# =============================================================================
# ICD9 DIAGNOSIS FEATURES
# =============================================================================

print("\n" + "="*80)
print("SECTION 3: CREATING ICD9 DIAGNOSIS FEATURES")
print("="*80)

# Verify diagnoses data structure
print(f"\n--- Diagnoses Data Structure ---")
print(f"Columns: {diagnoses.columns.tolist()}")
print(f"Sample:")
print(diagnoses.head(3))

# Build diagnosis lookup
print(f"\n--- Building Diagnosis Features ---")

# Feature 1: Number of diagnoses per admission
n_diagnoses_per_admission = diagnoses.groupby('HADM_ID').size()
print(f"\n1. Number of diagnoses per admission:")
print(f"   Mean: {n_diagnoses_per_admission.mean():.1f}")
print(f"   Median: {n_diagnoses_per_admission.median():.0f}")
print(f"   Max: {n_diagnoses_per_admission.max():.0f}")

train['n_diagnoses'] = train['hadm_id'].map(n_diagnoses_per_admission).fillna(0).astype(int)
test['n_diagnoses'] = test['hadm_id'].map(n_diagnoses_per_admission).fillna(0).astype(int)

print(f"   Train - Admissions with diagnoses: {(train['n_diagnoses'] > 0).sum()} ({(train['n_diagnoses'] > 0).mean()*100:.1f}%)")
print(f"   Test - Admissions with diagnoses: {(test['n_diagnoses'] > 0).sum()} ({(test['n_diagnoses'] > 0).mean()*100:.1f}%)")

# Feature 2: Primary diagnosis (SEQ_NUM = 1)
primary_diagnoses = diagnoses[diagnoses['SEQ_NUM'] == 1][['HADM_ID', 'ICD9_CODE']].set_index('HADM_ID')['ICD9_CODE']
print(f"\n2. Primary diagnoses:")
print(f"   Unique primary diagnoses: {primary_diagnoses.nunique()}")

train['primary_diagnosis_raw'] = train['hadm_id'].map(primary_diagnoses)
test['primary_diagnosis_raw'] = test['hadm_id'].map(primary_diagnoses)

print(f"   Train - Matched: {train['primary_diagnosis_raw'].notna().sum()} ({train['primary_diagnosis_raw'].notna().mean()*100:.1f}%)")
print(f"   Test - Matched: {test['primary_diagnosis_raw'].notna().sum()} ({test['primary_diagnosis_raw'].notna().mean()*100:.1f}%)")

# Feature 3: ICD9 category (first 3 characters)
def extract_icd9_category(code):
    """Extract first 3 characters from ICD9 code"""
    if pd.isna(code):
        return 'UNKNOWN'
    code_str = str(code).strip().replace('.', '')
    if len(code_str) >= 3:
        return code_str[:3]
    elif len(code_str) > 0:
        return code_str
    return 'UNKNOWN'

train['primary_diag_cat'] = train['primary_diagnosis_raw'].apply(extract_icd9_category)
test['primary_diag_cat'] = test['primary_diagnosis_raw'].apply(extract_icd9_category)

print(f"\n3. ICD9 categories (3-digit):")
print(f"   Unique categories: {train['primary_diag_cat'].nunique()}")
print(f"   Top 5 categories:")
for cat, count in train['primary_diag_cat'].value_counts().head().items():
    print(f"     {cat}: {count} ({count/len(train)*100:.1f}%)")

# Feature 4: Major disease category
def get_disease_category(code):
    """Map ICD9 code to major disease category"""
    if pd.isna(code):
        return 'UNKNOWN'
    
    code_str = str(code).strip().replace('.', '').replace(' ', '')
    if len(code_str) == 0:
        return 'UNKNOWN'
    
    first_char = code_str[0].upper()
    
    # ICD9 structure
    if first_char in ['0', '1']:
        return 'INFECTIOUS'
    elif first_char == '2':
        return 'NEOPLASM'
    elif first_char == '3':
        return 'ENDOCRINE'
    elif first_char == '4':
        return 'BLOOD'
    elif first_char == '5':
        return 'MENTAL'
    elif first_char in ['6', '7']:
        return 'NERVOUS'
    elif first_char == '8':
        return 'CIRCULATORY'
    elif first_char == '9':
        return 'RESPIRATORY'
    elif first_char == 'V':
        return 'V_CODE'
    elif first_char == 'E':
        return 'E_CODE'
    else:
        return 'OTHER'

train['disease_category'] = train['primary_diagnosis_raw'].apply(get_disease_category)
test['disease_category'] = test['primary_diagnosis_raw'].apply(get_disease_category)

print(f"\n4. Major disease categories:")
for cat, count in train['disease_category'].value_counts().items():
    print(f"   {cat}: {count} ({count/len(train)*100:.1f}%)")

# Feature 5: High-risk condition flags
print(f"\n5. High-risk condition flags:")

# Build efficient lookup: hadm_id -> set of all ICD9 codes
all_diagnoses_per_admission = diagnoses.groupby('HADM_ID')['ICD9_CODE'].apply(
    lambda x: set(str(code).replace('.', '').replace(' ', '') for code in x)
)

def check_condition_presence(hadm_id, code_patterns):
    """Check if any diagnosis matches the pattern"""
    if hadm_id not in all_diagnoses_per_admission.index:
        return 0
    
    codes = all_diagnoses_per_admission[hadm_id]
    
    for pattern in code_patterns:
        if any(code.startswith(pattern) for code in codes):
            return 1
    return 0

# Define condition patterns (common high-risk ICU conditions)
conditions = {
    'has_sepsis': ['99591', '99592', '78552'],  # Sepsis codes
    'has_heart_failure': ['428'],  # Heart failure
    'has_respiratory_failure': ['518'],  # Respiratory failure
    'has_aki': ['584'],  # Acute kidney injury
    'has_diabetes': ['250'],  # Diabetes
    'has_copd': ['491', '492', '496'],  # COPD
    'has_pneumonia': ['480', '481', '482', '483', '484', '485', '486']  # Pneumonia
}

for condition_name, patterns in conditions.items():
    train[condition_name] = train['hadm_id'].apply(lambda x: check_condition_presence(x, patterns))
    test[condition_name] = test['hadm_id'].apply(lambda x: check_condition_presence(x, patterns))
    
    count = train[condition_name].sum()
    print(f"   {condition_name}: {count} ({count/len(train)*100:.1f}%)")

print("\n" + "="*80)
print("✓ SECTION 3 COMPLETE - ICD9 features created")
print("="*80)


SECTION 3: CREATING ICD9 DIAGNOSIS FEATURES

--- Diagnoses Data Structure ---
Columns: ['SUBJECT_ID', 'HADM_ID', 'SEQ_NUM', 'ICD9_CODE']
Sample:
   SUBJECT_ID  HADM_ID  SEQ_NUM ICD9_CODE
0         256   108811      1.0     53240
1         256   108811      2.0     41071
2         256   108811      3.0     53560

--- Building Diagnosis Features ---

1. Number of diagnoses per admission:
   Mean: 11.0
   Median: 9
   Max: 39
   Train - Admissions with diagnoses: 20885 (100.0%)
   Test - Admissions with diagnoses: 5221 (100.0%)

2. Primary diagnoses:
   Unique primary diagnoses: 2789
   Train - Matched: 20885 (100.0%)
   Test - Matched: 5221 (100.0%)

3. ICD9 categories (3-digit):
   Unique categories: 530
   Top 5 categories:
     038: 1595 (7.6%)
     414: 1115 (5.3%)
     410: 948 (4.5%)
     424: 744 (3.6%)
     428: 686 (3.3%)

4. Major disease categories:
   BLOOD: 7507 (35.9%)
   MENTAL: 3912 (18.7%)
   INFECTIOUS: 3208 (15.4%)
   CIRCULATORY: 1795 (8.6%)
   RESPIRATORY: 1578 (7.6

---
<a id='section4'></a>
## 4. Remove Leakage Columns

Removing columns that would cause data leakage:
- `DISCHTIME`, `DEATHTIME`, `DOD`: Death/discharge information (target leakage)
- `LOS`: Length of stay (correlated with outcome, not available at prediction time)
- ID columns after extracting features

In [8]:
# =============================================================================
# DROP LEAKAGE COLUMNS
# =============================================================================

print("\n" + "="*80)
print("SECTION 4: REMOVING LEAKAGE COLUMNS")
print("="*80)

# Columns that leak information about the target
leakage_columns = [
    'DISCHTIME',      # Discharge time (only known after outcome)
    'DEATHTIME',      # Death time (IS the target!)
    'DOD',            # Date of death (IS the target!)
    'LOS',            # Length of stay (correlated with outcome)
    'Diff',           # Time difference (likely leakage)
    'ADMITTIME',      # Already used for history features
]

# IDs - we've extracted all useful info
id_columns = [
    'icustay_id',     # Already saved as test_ids
    'subject_id',     # Used for history features
    'hadm_id',        # Used for diagnosis matching
]

# Raw diagnosis column - we've extracted features
diagnosis_columns = [
    'primary_diagnosis_raw'
]

all_columns_to_drop = leakage_columns + id_columns + diagnosis_columns

print(f"\n--- Columns to Drop ---")
for col in all_columns_to_drop:
    train_has = "✓" if col in train.columns else "✗"
    test_has = "✓" if col in test.columns else "✗"
    print(f"  {col:25s} Train:{train_has}  Test:{test_has}")

# Drop from train
train_clean = train.drop(columns=[c for c in all_columns_to_drop if c in train.columns], errors='ignore')

# Drop from test  
test_clean = test.drop(columns=[c for c in all_columns_to_drop if c in test.columns], errors='ignore')

print(f"\n--- Shape Changes ---")
print(f"  Train: {train.shape} → {train_clean.shape}")
print(f"  Test:  {test.shape} → {test_clean.shape}")

# Separate target from train
print(f"\n--- Separating Target ---")
if 'HOSPITAL_EXPIRE_FLAG' not in train_clean.columns:
    print("  ❌ ERROR: Target column not found!")
    raise ValueError("HOSPITAL_EXPIRE_FLAG column missing!")

y = train_clean['HOSPITAL_EXPIRE_FLAG'].copy()
X = train_clean.drop('HOSPITAL_EXPIRE_FLAG', axis=1)
X_test = test_clean.copy()

print(f"  ✓ Target separated")
print(f"  ✓ y shape: {y.shape}")
print(f"  ✓ X shape: {X.shape}")
print(f"  ✓ X_test shape: {X_test.shape}")

# Validate target
print(f"\n--- Target Validation ---")
print(f"  Target name: HOSPITAL_EXPIRE_FLAG")
print(f"  Unique values: {y.unique()}")
print(f"  Mortality rate: {y.mean():.3f} ({y.sum()}/{len(y)})")
print(f"  Class balance: 0={y.value_counts()[0]}, 1={y.value_counts()[1]}")

# Verify train and test have same columns (except target)
print(f"\n--- Column Consistency Check ---")
train_cols = set(X.columns)
test_cols = set(X_test.columns)

if train_cols == test_cols:
    print(f"  ✓ Train and test have identical columns ({len(train_cols)} columns)")
else:
    cols_only_in_train = train_cols - test_cols
    cols_only_in_test = test_cols - train_cols
    if cols_only_in_train:
        print(f"  ⚠️ Columns only in train: {cols_only_in_train}")
    if cols_only_in_test:
        print(f"  ⚠️ Columns only in test: {cols_only_in_test}")

print("\n" + "="*80)
print("✓ SECTION 4 COMPLETE - Leakage columns removed")
print("="*80)


SECTION 4: REMOVING LEAKAGE COLUMNS

--- Columns to Drop ---
  DISCHTIME                 Train:✓  Test:✗
  DEATHTIME                 Train:✓  Test:✗
  DOD                       Train:✓  Test:✗
  LOS                       Train:✓  Test:✗
  Diff                      Train:✓  Test:✓
  ADMITTIME                 Train:✓  Test:✓
  icustay_id                Train:✓  Test:✓
  subject_id                Train:✓  Test:✓
  hadm_id                   Train:✓  Test:✓
  primary_diagnosis_raw     Train:✓  Test:✓

--- Shape Changes ---
  Train: (20885, 58) → (20885, 48)
  Test:  (5221, 53) → (5221, 47)

--- Separating Target ---
  ✓ Target separated
  ✓ y shape: (20885,)
  ✓ X shape: (20885, 47)
  ✓ X_test shape: (5221, 47)

--- Target Validation ---
  Target name: HOSPITAL_EXPIRE_FLAG
  Unique values: [0 1]
  Mortality rate: 0.112 (2345/20885)
  Class balance: 0=18540, 1=2345

--- Column Consistency Check ---
  ✓ Train and test have identical columns (47 columns)

✓ SECTION 4 COMPLETE - Leakage column

---
<a id='section5'></a>
## 5. Age Calculation

Converting DOB to age at admission. Note: MIMIC-III shifts DOB ~300 years backward for patients >89 years old for de-identification - we handle this by detecting and imputing these cases.

In [9]:
# =============================================================================
# CONVERT DOB TO AGE
# =============================================================================

print("\n" + "="*80)
print("SECTION 5: CONVERTING DOB TO AGE")
print("="*80)

if 'DOB' not in X.columns:
    print("  ⚠️ DOB column not found, skipping age calculation")
else:
    print("\n--- Loading original data for ADMITTIME ---")
    
    # Convert to datetime
    print("\n--- Converting dates ---")
    dob_train = pd.to_datetime(X['DOB'], errors='coerce')
    dob_test = pd.to_datetime(X_test['DOB'], errors='coerce')
    admit_train = pd.to_datetime(train_raw['ADMITTIME'], errors='coerce')
    admit_test = pd.to_datetime(test_raw['ADMITTIME'], errors='coerce')
    
    print(f"  Train - DOB parsed: {dob_train.notna().sum()}/{len(dob_train)}")
    print(f"  Train - ADMITTIME parsed: {admit_train.notna().sum()}/{len(admit_train)}")
    
    # Calculate age in years (using year-based calculation to avoid overflow)
    print("\n--- Calculating ages ---")
    
    # Year-based age calculation (avoids integer overflow with shifted DOBs)
    age_train = admit_train.dt.year - dob_train.dt.year
    age_test = admit_test.dt.year - dob_test.dt.year
    
    # Adjust for birthday not yet occurred
    birthday_not_passed_train = (admit_train.dt.month < dob_train.dt.month) | \
                                 ((admit_train.dt.month == dob_train.dt.month) & 
                                  (admit_train.dt.day < dob_train.dt.day))
    birthday_not_passed_test = (admit_test.dt.month < dob_test.dt.month) | \
                                ((admit_test.dt.month == dob_test.dt.month) & 
                                 (admit_test.dt.day < dob_test.dt.day))
    
    age_train = age_train - birthday_not_passed_train.astype(int)
    age_test = age_test - birthday_not_passed_test.astype(int)
    
    X['age'] = age_train.values
    X_test['age'] = age_test.values
    
    print(f"  ✓ Ages calculated")
    
    # Check for invalid ages (MIMIC-III shifts DOB for patients >89)
    print(f"\n--- Age Distribution (Before Cleaning) ---")
    print(f"  Train:")
    print(f"    Min: {X['age'].min():.1f}")
    print(f"    Max: {X['age'].max():.1f}")
    print(f"    Mean: {X['age'].mean():.1f}")
    print(f"    Missing: {X['age'].isna().sum()}")
    
    # Clean invalid ages
    print(f"\n--- Cleaning Invalid Ages ---")
    
    # Invalid: negative or > 120 (MIMIC shifts DOB for >89 patients)
    invalid_train = (X['age'] < 0) | (X['age'] > 120) | X['age'].isna()
    invalid_test = (X_test['age'] < 0) | (X_test['age'] > 120) | X_test['age'].isna()
    
    print(f"  Train - Invalid ages: {invalid_train.sum()}")
    print(f"  Test - Invalid ages: {invalid_test.sum()}")
    
    # Set invalid to NaN
    X.loc[invalid_train, 'age'] = np.nan
    X_test.loc[invalid_test, 'age'] = np.nan
    
    # Impute missing ages with median (for elderly patients with shifted DOB)
    age_median = X['age'].median()
    n_missing_train = X['age'].isna().sum()
    n_missing_test = X_test['age'].isna().sum()
    
    X['age'].fillna(age_median, inplace=True)
    X_test['age'].fillna(age_median, inplace=True)
    
    print(f"  ✓ Imputed {n_missing_train} train + {n_missing_test} test missing ages with median: {age_median:.1f}")
    
    # Clip to valid range
    X['age'] = X['age'].clip(0, 120)
    X_test['age'] = X_test['age'].clip(0, 120)
    
    # Final age distribution
    print(f"\n--- Age Distribution (After Cleaning) ---")
    print(f"  Train:")
    print(f"    Range: {X['age'].min():.1f} - {X['age'].max():.1f} years")
    print(f"    Mean: {X['age'].mean():.1f} years")
    print(f"    Std: {X['age'].std():.1f} years")
    
    # Drop DOB column
    X = X.drop('DOB', axis=1)
    X_test = X_test.drop('DOB', axis=1)
    
    print(f"\n  ✓ Dropped DOB column")
    
    # Validation
    print(f"\n--- Validation ---")
    assert X['age'].notna().all(), "ERROR: Still have NaN ages in train!"
    assert X_test['age'].notna().all(), "ERROR: Still have NaN ages in test!"
    assert (X['age'] >= 0).all() and (X['age'] <= 120).all(), "ERROR: Invalid ages in train!"
    assert (X_test['age'] >= 0).all() and (X_test['age'] <= 120).all(), "ERROR: Invalid ages in test!"
    
    print(f"  ✓ All validation checks passed")

print("\n" + "="*80)
print("✓ SECTION 5 COMPLETE - DOB converted to age")
print("="*80)


SECTION 5: CONVERTING DOB TO AGE

--- Loading original data for ADMITTIME ---

--- Converting dates ---
  Train - DOB parsed: 20885/20885
  Train - ADMITTIME parsed: 20885/20885

--- Calculating ages ---


ValueError: Can only compare identically-labeled Series objects

---
<a id='section6'></a>
## 6. Missing Value Imputation

Handling missing values:
- Numeric features: Median imputation
- Categorical features: Most frequent value imputation

In [ ]:
# =============================================================================
# FEATURE TYPE IDENTIFICATION & IMPUTATION
# =============================================================================

print("\n" + "="*80)
print("SECTION 6: FEATURE TYPE IDENTIFICATION & IMPUTATION")
print("="*80)

# Identify feature types
print("\n--- Identifying Feature Types ---")

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"  Numeric features: {len(numeric_features)}")
print(f"  Categorical features: {len(categorical_features)}")

# Show sample of each type
print(f"\n--- Sample Features ---")
print(f"  Numeric (first 10): {numeric_features[:10]}")
print(f"  Categorical: {categorical_features}")

# Check missing values
print(f"\n--- Missing Value Analysis ---")

missing_numeric = X[numeric_features].isnull().sum()
missing_numeric = missing_numeric[missing_numeric > 0].sort_values(ascending=False)

if len(missing_numeric) > 0:
    print(f"  Numeric features with missing values:")
    for feat, count in missing_numeric.head(10).items():
        pct = count / len(X) * 100
        print(f"    {feat:30s} {count:6d} ({pct:5.1f}%)")
else:
    print(f"  ✓ No missing values in numeric features")

missing_categorical = X[categorical_features].isnull().sum()
missing_categorical = missing_categorical[missing_categorical > 0].sort_values(ascending=False)

if len(missing_categorical) > 0:
    print(f"\n  Categorical features with missing values:")
    for feat, count in missing_categorical.items():
        pct = count / len(X) * 100
        print(f"    {feat:30s} {count:6d} ({pct:5.1f}%)")
else:
    print(f"  ✓ No missing values in categorical features")

# Imputation
print(f"\n--- Imputation Strategy ---")

# Numeric: median imputation
if len(numeric_features) > 0:
    print(f"  Numeric features: Median imputation")
    numeric_imputer = SimpleImputer(strategy='median')
    X[numeric_features] = numeric_imputer.fit_transform(X[numeric_features])
    X_test[numeric_features] = numeric_imputer.transform(X_test[numeric_features])
    print(f"  ✓ Imputed {len(numeric_features)} numeric features")

# Categorical: most frequent imputation
if len(categorical_features) > 0:
    print(f"  Categorical features: Most frequent imputation")
    categorical_imputer = SimpleImputer(strategy='most_frequent')
    X[categorical_features] = categorical_imputer.fit_transform(X[categorical_features])
    X_test[categorical_features] = categorical_imputer.transform(X_test[categorical_features])
    print(f"  ✓ Imputed {len(categorical_features)} categorical features")

# Verify no missing values remain
print(f"\n--- Post-Imputation Validation ---")

train_missing = X.isnull().sum().sum()
test_missing = X_test.isnull().sum().sum()

print(f"  Train missing values: {train_missing}")
print(f"  Test missing values: {test_missing}")

if train_missing == 0 and test_missing == 0:
    print(f"  ✓ No missing values remain")

print("\n" + "="*80)
print("✓ SECTION 6 COMPLETE - Features imputed")
print("="*80)

---
<a id='section7'></a>
## 7. Categorical Encoding

Encoding strategy:
- High-cardinality features (ICD9, DIAGNOSIS): Target encoding
- Medium-cardinality features (ETHNICITY, RELIGION): Group then one-hot encode
- Low-cardinality features (GENDER, ADMISSION_TYPE, etc.): One-hot encoding

In [ ]:
# =============================================================================
# ENCODE CATEGORICAL FEATURES
# =============================================================================

print("\n" + "="*80)
print("SECTION 7: ENCODING CATEGORICAL FEATURES")
print("="*80)

print("\n--- Categorical Feature Analysis ---")

for cat_col in categorical_features:
    n_unique = X[cat_col].nunique()
    print(f"  {cat_col:25s} {n_unique:4d} unique values")
    
    # Show distribution for low-cardinality features
    if n_unique <= 10:
        print(f"    Distribution:")
        for val, count in X[cat_col].value_counts().head(5).items():
            print(f"      {str(val):30s} {count:6d} ({count/len(X)*100:5.1f}%)")

# Strategy for each categorical feature
print("\n" + "="*80)
print("ENCODING STRATEGY")
print("="*80)

# --- 1. ICD9_diagnosis: Target encode ---
print("\n1. ICD9_diagnosis → Target encode")

if 'ICD9_diagnosis' in X.columns:
    def extract_icd9_cat(code):
        if pd.isna(code):
            return 'UNKNOWN'
        code_str = str(code).strip().replace('.', '')
        if len(code_str) >= 3:
            return code_str[:3]
        elif len(code_str) > 0:
            return code_str
        return 'UNKNOWN'
    
    X['ICD9_cat'] = X['ICD9_diagnosis'].apply(extract_icd9_cat)
    X_test['ICD9_cat'] = X_test['ICD9_diagnosis'].apply(extract_icd9_cat)
    
    # Target encode
    encoding_map = y.groupby(X['ICD9_cat']).mean().to_dict()
    global_mean = y.mean()
    
    X['ICD9_encoded'] = X['ICD9_cat'].map(encoding_map).fillna(global_mean)
    X_test['ICD9_encoded'] = X_test['ICD9_cat'].map(encoding_map).fillna(global_mean)
    
    # Drop original columns
    X = X.drop(['ICD9_diagnosis', 'ICD9_cat'], axis=1)
    X_test = X_test.drop(['ICD9_diagnosis', 'ICD9_cat'], axis=1)
    
    print(f"   ✓ Encoded {len(encoding_map)} categories")
    print(f"   Mortality range: {min(encoding_map.values()):.3f} - {max(encoding_map.values()):.3f}")

# --- 2. primary_diag_cat: Target encode ---
print("\n2. primary_diag_cat → Target encode")

if 'primary_diag_cat' in X.columns:
    encoding_map = y.groupby(X['primary_diag_cat']).mean().to_dict()
    global_mean = y.mean()
    
    X['primary_diag_encoded'] = X['primary_diag_cat'].map(encoding_map).fillna(global_mean)
    X_test['primary_diag_encoded'] = X_test['primary_diag_cat'].map(encoding_map).fillna(global_mean)
    
    # Drop original column
    X = X.drop('primary_diag_cat', axis=1)
    X_test = X_test.drop('primary_diag_cat', axis=1)
    
    print(f"   ✓ Encoded {len(encoding_map)} categories")
    print(f"   Mortality range: {min(encoding_map.values()):.3f} - {max(encoding_map.values()):.3f}")

# --- 3. DIAGNOSIS: Drop (free text, already have ICD9 codes) ---
print("\n3. DIAGNOSIS → Drop")

if 'DIAGNOSIS' in X.columns:
    print(f"   Unique values: {X['DIAGNOSIS'].nunique()}")
    X = X.drop('DIAGNOSIS', axis=1)
    X_test = X_test.drop('DIAGNOSIS', axis=1)
    print("   ✓ Dropping (free text, already have ICD9 codes)")

# --- 4. Group categorical features ---
print("\n4. Grouping categorical features")

# Update categorical features list
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# ETHNICITY
if 'ETHNICITY' in categorical_features:
    def group_ethnicity(ethnicity):
        if pd.isna(ethnicity):
            return 'UNKNOWN'
        ethnicity = str(ethnicity).upper()
        if 'WHITE' in ethnicity:
            return 'WHITE'
        elif 'BLACK' in ethnicity or 'AFRICAN' in ethnicity:
            return 'BLACK'
        elif 'HISPANIC' in ethnicity or 'LATINO' in ethnicity:
            return 'HISPANIC'
        elif 'ASIAN' in ethnicity:
            return 'ASIAN'
        elif any(x in ethnicity for x in ['UNKNOWN', 'UNABLE', 'DECLINED', 'NOT SPECIFIED']):
            return 'UNKNOWN'
        else:
            return 'OTHER'
    
    X['ETHNICITY'] = X['ETHNICITY'].apply(group_ethnicity)
    X_test['ETHNICITY'] = X_test['ETHNICITY'].apply(group_ethnicity)
    
    print(f"   ETHNICITY: {X['ETHNICITY'].nunique()} categories")

# RELIGION
if 'RELIGION' in categorical_features:
    def group_religion(religion):
        if pd.isna(religion):
            return 'UNKNOWN'
        religion = str(religion).upper()
        if 'CATHOLIC' in religion:
            return 'CATHOLIC'
        elif any(x in religion for x in ['PROTESTANT', 'EPISCOPALIAN', 'QUAKER']):
            return 'PROTESTANT'
        elif 'JEWISH' in religion or 'HEBREW' in religion:
            return 'JEWISH'
        elif any(x in religion for x in ['UNOBTAINABLE', 'NOT SPECIFIED', 'UNKNOWN']):
            return 'UNKNOWN'
        else:
            return 'OTHER'
    
    X['RELIGION'] = X['RELIGION'].apply(group_religion)
    X_test['RELIGION'] = X_test['RELIGION'].apply(group_religion)
    
    print(f"   RELIGION: {X['RELIGION'].nunique()} categories")

# MARITAL_STATUS
if 'MARITAL_STATUS' in categorical_features:
    def group_marital_status(status):
        if pd.isna(status):
            return 'UNKNOWN'
        status = str(status).upper()
        if 'MARRIED' in status or 'LIFE PARTNER' in status:
            return 'MARRIED'
        elif 'SINGLE' in status:
            return 'SINGLE'
        elif 'WIDOWED' in status:
            return 'WIDOWED'
        elif 'DIVORCED' in status or 'SEPARATED' in status:
            return 'DIVORCED_SEPARATED'
        else:
            return 'UNKNOWN'
    
    X['MARITAL_STATUS'] = X['MARITAL_STATUS'].apply(group_marital_status)
    X_test['MARITAL_STATUS'] = X_test['MARITAL_STATUS'].apply(group_marital_status)
    
    print(f"   MARITAL_STATUS: {X['MARITAL_STATUS'].nunique()} categories")

# --- 5. One-hot encode remaining categoricals ---
print("\n5. One-hot encoding remaining features")

remaining_categorical = X.select_dtypes(include=['object']).columns.tolist()
print(f"   Features to one-hot encode: {remaining_categorical}")

if len(remaining_categorical) > 0:
    # Combine train and test to ensure same columns
    X_combined = pd.concat([X, X_test], keys=['train', 'test'])
    
    # One-hot encode
    X_encoded = pd.get_dummies(
        X_combined, 
        columns=remaining_categorical, 
        drop_first=True,
        dtype=int
    )
    
    # Split back
    X = X_encoded.xs('train')
    X_test = X_encoded.xs('test')
    
    n_new_features = len([col for col in X.columns if any(cat in col for cat in remaining_categorical)])
    print(f"   ✓ Created {n_new_features} binary features")

# --- Validation ---
print("\n" + "="*80)
print("ENCODING VALIDATION")
print("="*80)

# Check for remaining object columns
object_cols_train = X.select_dtypes(include=['object']).columns.tolist()

if object_cols_train:
    print(f"  ⚠️ Warning: Still have object columns: {object_cols_train}")
else:
    print(f"  ✓ No object columns remain - all categorical features encoded")

# Check train/test consistency
if list(X.columns) == list(X_test.columns):
    print(f"  ✓ Train and test have identical columns: {X.shape[1]}")
else:
    print(f"  ❌ ERROR: Train/test column mismatch after encoding!")

print(f"\n  Final feature count: {X.shape[1]}")

print("\n" + "="*80)
print("✓ SECTION 7 COMPLETE - Categorical features encoded")
print("="*80)

---
<a id='section8'></a>
## 8. Medical Feature Engineering

Creating clinically-relevant derived features:
- Shock indices (Heart Rate / Blood Pressure ratios)
- Vital sign ranges and abnormality indicators
- Age-based risk features
- Composite severity score

In [ ]:
# =============================================================================
# MEDICAL FEATURE ENGINEERING
# =============================================================================

print("\n" + "="*80)
print("SECTION 8: MEDICAL FEATURE ENGINEERING")
print("="*80)

original_feature_count = X.shape[1]

print("\n--- Creating Vital Sign Features ---")

# Blood Pressure Features
if all(col in X.columns for col in ['SysBP_Mean', 'DiasBP_Mean']):
    X['PulsePressure'] = X['SysBP_Mean'] - X['DiasBP_Mean']
    X_test['PulsePressure'] = X_test['SysBP_Mean'] - X_test['DiasBP_Mean']
    print("  ✓ Pulse pressure")

if all(col in X.columns for col in ['SysBP_Min', 'SysBP_Max']):
    X['SysBP_Range'] = X['SysBP_Max'] - X['SysBP_Min']
    X_test['SysBP_Range'] = X_test['SysBP_Max'] - X_test['SysBP_Min']
    print("  ✓ Systolic BP range")

# Shock Indices (critical for ICU mortality prediction)
if all(col in X.columns for col in ['HeartRate_Mean', 'SysBP_Mean']):
    X['ShockIndex'] = (X['HeartRate_Mean'] / (X['SysBP_Mean'] + 1)).clip(0, 3)
    X_test['ShockIndex'] = (X_test['HeartRate_Mean'] / (X_test['SysBP_Mean'] + 1)).clip(0, 3)
    print("  ✓ Shock index (clipped 0-3)")

if all(col in X.columns for col in ['HeartRate_Mean', 'MeanBP_Mean']):
    X['ModifiedShockIndex'] = (X['HeartRate_Mean'] / (X['MeanBP_Mean'] + 1)).clip(0, 3)
    X_test['ModifiedShockIndex'] = (X_test['HeartRate_Mean'] / (X_test['MeanBP_Mean'] + 1)).clip(0, 3)
    print("  ✓ Modified shock index (clipped 0-3)")

# Respiratory Features
if 'SpO2_Min' in X.columns:
    X['Hypoxemia'] = (X['SpO2_Min'] < 90).astype(int)
    X_test['Hypoxemia'] = (X_test['SpO2_Min'] < 90).astype(int)
    print("  ✓ Hypoxemia indicator")

if 'RespRate_Mean' in X.columns:
    X['RespRate_Abnormal'] = ((X['RespRate_Mean'] < 12) | (X['RespRate_Mean'] > 20)).astype(int)
    X_test['RespRate_Abnormal'] = ((X_test['RespRate_Mean'] < 12) | (X_test['RespRate_Mean'] > 20)).astype(int)
    print("  ✓ Abnormal respiratory rate")

# Temperature Features
if 'TempC_Max' in X.columns:
    X['Fever'] = (X['TempC_Max'] > 38).astype(int)
    X_test['Fever'] = (X_test['TempC_Max'] > 38).astype(int)
    print("  ✓ Fever indicator")

if 'TempC_Min' in X.columns:
    X['Hypothermia'] = (X['TempC_Min'] < 36).astype(int)
    X_test['Hypothermia'] = (X_test['TempC_Min'] < 36).astype(int)
    print("  ✓ Hypothermia indicator")

if all(col in X.columns for col in ['TempC_Min', 'TempC_Max']):
    X['Temp_Range'] = X['TempC_Max'] - X['TempC_Min']
    X_test['Temp_Range'] = X_test['TempC_Max'] - X_test['TempC_Min']
    print("  ✓ Temperature range")

# Glucose Features  
if 'Glucose_Max' in X.columns:
    X['Hyperglycemia'] = (X['Glucose_Max'] > 180).astype(int)
    X_test['Hyperglycemia'] = (X_test['Glucose_Max'] > 180).astype(int)
    print("  ✓ Hyperglycemia indicator")

if 'Glucose_Min' in X.columns:
    X['Hypoglycemia'] = (X['Glucose_Min'] < 70).astype(int)
    X_test['Hypoglycemia'] = (X_test['Glucose_Min'] < 70).astype(int)
    print("  ✓ Hypoglycemia indicator")

if all(col in X.columns for col in ['Glucose_Min', 'Glucose_Max']):
    X['Glucose_Range'] = X['Glucose_Max'] - X['Glucose_Min']
    X_test['Glucose_Range'] = X_test['Glucose_Max'] - X_test['Glucose_Min']
    print("  ✓ Glucose variability")

# Heart Rate Range
if all(col in X.columns for col in ['HeartRate_Min', 'HeartRate_Max']):
    X['HeartRate_Range'] = X['HeartRate_Max'] - X['HeartRate_Min']
    X_test['HeartRate_Range'] = X_test['HeartRate_Max'] - X_test['HeartRate_Min']
    print("  ✓ Heart rate range")

# Age Features
print("\n--- Creating Age-Based Features ---")

if 'age' in X.columns:
    # Elderly indicator
    X['Elderly'] = (X['age'] > 65).astype(int)
    X_test['Elderly'] = (X_test['age'] > 65).astype(int)
    print("  ✓ Elderly indicator (>65)")
    
    # Age squared (non-linear effects)
    X['age_squared'] = X['age'] ** 2
    X_test['age_squared'] = X_test['age'] ** 2
    print("  ✓ Age squared")
    
    # Age risk groups
    age_bins = [0, 18, 45, 65, 80, 120]
    age_labels = ['pediatric', 'young_adult', 'middle_age', 'elderly', 'very_old']
    
    X['age_group'] = pd.cut(X['age'], bins=age_bins, labels=age_labels)
    X_test['age_group'] = pd.cut(X_test['age'], bins=age_bins, labels=age_labels)
    
    # One-hot encode age groups
    X_combined = pd.concat([X, X_test], keys=['train', 'test'])
    X_encoded = pd.get_dummies(X_combined, columns=['age_group'], drop_first=True, prefix='age', dtype=int)
    X = X_encoded.xs('train')
    X_test = X_encoded.xs('test')
    
    print("  ✓ Age risk groups (one-hot encoded)")

# Composite Severity Score
print("\n--- Creating Composite Severity Score ---")

severity_components = []
severity_components_test = []

if 'ShockIndex' in X.columns:
    severity_components.append((X['ShockIndex'] > 0.9).astype(int))
    severity_components_test.append((X_test['ShockIndex'] > 0.9).astype(int))
if 'Hypoxemia' in X.columns:
    severity_components.append(X['Hypoxemia'])
    severity_components_test.append(X_test['Hypoxemia'])
if 'RespRate_Abnormal' in X.columns:
    severity_components.append(X['RespRate_Abnormal'])
    severity_components_test.append(X_test['RespRate_Abnormal'])
if 'Fever' in X.columns:
    severity_components.append(X['Fever'])
    severity_components_test.append(X_test['Fever'])
if 'Hypothermia' in X.columns:
    severity_components.append(X['Hypothermia'])
    severity_components_test.append(X_test['Hypothermia'])

if severity_components:
    X['Severity_Score'] = sum(severity_components)
    X_test['Severity_Score'] = sum(severity_components_test)
    print(f"  ✓ Severity score (0-{len(severity_components)})")
    print(f"    Distribution: {X['Severity_Score'].value_counts().sort_index().to_dict()}")

# Summary
print(f"\n--- Feature Engineering Summary ---")
print(f"  Features before: {original_feature_count}")
print(f"  Features after: {X.shape[1]}")
print(f"  Features added: {X.shape[1] - original_feature_count}")

print("\n" + "="*80)
print("✓ SECTION 8 COMPLETE - Medical features engineered")
print("="*80)

---
<a id='section9'></a>
## 9. Feature Scaling

Applying StandardScaler to continuous features only:
- Binary features (0/1): NOT scaled
- Count features: NOT scaled  
- Continuous vitals/measurements: Scaled to mean=0, std=1

In [ ]:
# =============================================================================
# FEATURE SCALING
# =============================================================================

print("\n" + "="*80)
print("SECTION 9: FEATURE SCALING")
print("="*80)

print("\n--- Identifying Features to Scale ---")

# Get all numeric columns
all_numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print(f"  Total numeric columns: {len(all_numeric_cols)}")

# Features that should NOT be scaled (binary, count, ordinal)
binary_features_list = [
    # Medical binary indicators
    'Hypoxemia', 'RespRate_Abnormal', 'Fever', 'Hypothermia',
    'Hyperglycemia', 'Hypoglycemia', 'Elderly',
    # Hospital history - binary flags
    'is_first_icu_visit', 'is_frequent_flyer',
    # Condition flags - binary
    'has_sepsis', 'has_heart_failure', 'has_respiratory_failure',
    'has_aki', 'has_diabetes', 'has_copd', 'has_pneumonia'
]

# Count features - DO NOT SCALE
count_features = ['n_previous_icu_stays', 'n_diagnoses']

# Ordinal features - DO NOT SCALE
ordinal_features = ['Severity_Score']

# One-hot encoded features (binary)
one_hot_features = [col for col in X.columns if '_' in col and X[col].nunique() <= 2]

print(f"  Binary indicator features: {len(binary_features_list)}")
print(f"  Count features: {len(count_features)}")
print(f"  One-hot encoded features: {len(one_hot_features)}")

# Combine ALL features to exclude from scaling
exclude_from_scaling = list(set(
    binary_features_list + 
    count_features + 
    ordinal_features + 
    one_hot_features
))
exclude_from_scaling = [col for col in exclude_from_scaling if col in all_numeric_cols]

# Features to scale = numeric features - excluded features
features_to_scale = [col for col in all_numeric_cols if col not in exclude_from_scaling]

print(f"\n  Features to scale: {len(features_to_scale)}")
print(f"  Features to keep unscaled: {len(exclude_from_scaling)}")

# Show sample
print(f"\n--- Sample Features ---")
print(f"  Scaling (first 10): {features_to_scale[:10]}")
print(f"  Not scaling (first 10): {exclude_from_scaling[:10]}")

# Scale continuous features only
print(f"\n--- Applying StandardScaler ---")

scaler = StandardScaler()

X[features_to_scale] = scaler.fit_transform(X[features_to_scale])
X_test[features_to_scale] = scaler.transform(X_test[features_to_scale])

print(f"  ✓ Scaled {len(features_to_scale)} continuous features")
print(f"  ✓ Left {len(exclude_from_scaling)} features unscaled")

# Validation
print(f"\n--- Scaling Validation ---")

# Check scaled features have mean ≈ 0, std ≈ 1
print(f"\n  Checking scaled feature statistics (sample):")
sample_features = features_to_scale[:5]
for feat in sample_features:
    mean = X[feat].mean()
    std = X[feat].std()
    print(f"    {feat:30s} mean={mean:7.4f}, std={std:7.4f}")

# Check binary features remain 0/1
print(f"\n  Checking binary features remain 0/1:")
binary_check_passed = True
for feat in binary_features_list:
    if feat in X.columns:
        unique_vals = set(X[feat].unique())
        if not unique_vals.issubset({0, 1, 0.0, 1.0}):
            print(f"    ❌ {feat}: values are {sorted(unique_vals)[:5]}")
            binary_check_passed = False
if binary_check_passed:
    print(f"    ✓ All binary features remain 0/1")

# Check for NaN or Inf
print(f"\n  Checking for invalid values:")
nan_count = X.isnull().sum().sum()
inf_count = np.isinf(X.select_dtypes(include=[np.number])).sum().sum()
print(f"    NaN values: {nan_count}")
print(f"    Infinite values: {inf_count}")
if nan_count == 0 and inf_count == 0:
    print(f"    ✓ No invalid values")

print("\n" + "="*80)
print("✓ SECTION 9 COMPLETE - Features scaled")
print("="*80)

---
<a id='section10'></a>
## 10. Final Validation

Comprehensive validation before model training.

In [ ]:
# =============================================================================
# FINAL VALIDATION
# =============================================================================

print("\n" + "="*80)
print("SECTION 10: FINAL VALIDATION")
print("="*80)

validation_passed = True
issues = []

# Check 1: Shape consistency
print("\n--- Shape Consistency ---")
print(f"  X_train: {X.shape}")
print(f"  y_train: {y.shape}")
print(f"  X_test: {X_test.shape}")
print(f"  test_ids: {len(test_ids)}")

if X.shape[0] != len(y):
    issues.append("X_train and y_train have different number of samples!")
    validation_passed = False
if X_test.shape[0] != len(test_ids):
    issues.append("X_test and test_ids have different number of samples!")
    validation_passed = False
if X.shape[1] != X_test.shape[1]:
    issues.append(f"Feature count mismatch: train={X.shape[1]}, test={X_test.shape[1]}")
    validation_passed = False

# Check 2: No missing values
print("\n--- Missing Values ---")
train_missing = X.isnull().sum().sum()
test_missing = X_test.isnull().sum().sum()
print(f"  Train: {train_missing}")
print(f"  Test: {test_missing}")

if train_missing > 0 or test_missing > 0:
    issues.append(f"Missing values: train={train_missing}, test={test_missing}")
    validation_passed = False

# Check 3: Column alignment
print("\n--- Column Alignment ---")
if list(X.columns) == list(X_test.columns):
    print("  ✓ Train and test columns are aligned")
else:
    issues.append("Column alignment issue between train and test!")
    validation_passed = False

# Check 4: Target variable
print("\n--- Target Variable ---")
print(f"  Unique values: {sorted(y.unique())}")
print(f"  Class 0: {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)")
print(f"  Class 1: {(y == 1).sum()} ({(y == 1).mean()*100:.1f}%)")

if set(y.unique()) != {0, 1}:
    issues.append("Target should be binary (0, 1)!")
    validation_passed = False

# Check 5: Test IDs alignment
print("\n--- Test IDs ---")
print(f"  Number of test IDs: {len(test_ids)}")
print(f"  First 5 IDs: {test_ids.head().tolist()}")

# Final verdict
print("\n" + "="*80)
if validation_passed and len(issues) == 0:
    print("✅ ALL VALIDATION CHECKS PASSED!")
    print("="*80)
    print("\n🎉 Data is ready for modeling!")
else:
    print("🚨 VALIDATION ISSUES FOUND")
    print("="*80)
    for i, issue in enumerate(issues, 1):
        print(f"  {i}. {issue}")
    print("\n⚠️ Review and fix issues before training models!")

---
<a id='section11'></a>
## 11. Model Training - Gradient Boosting with Optuna

Training a Gradient Boosting Classifier with Optuna hyperparameter optimization:
- Objective: Maximize ROC-AUC with CV - 0.5*std (penalize high variance)
- 5-fold stratified cross-validation
- Focus on generalization to minimize CV-Kaggle gap

In [ ]:
# =============================================================================
# GRADIENT BOOSTING + OPTUNA OPTIMIZATION
# =============================================================================

print("\n" + "="*80)
print("SECTION 11: GRADIENT BOOSTING + OPTUNA")
print("="*80)

# Define objective function for Optuna
def objective(trial):
    """
    Gradient Boosting parameter search.
    Focus: Maximum generalization (minimize CV-Kaggle gap)
    """
    
    params = {
        # Number of trees
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        
        # Learning rate - LOWER for better generalization
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        
        # Tree depth - SHALLOW for generalization
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        
        # Minimum samples - HIGH values for regularization
        'min_samples_split': trial.suggest_int('min_samples_split', 10, 100),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 5, 50),
        
        # Sampling - helps prevent overfitting
        'subsample': trial.suggest_float('subsample', 0.5, 0.95),
        
        # Feature sampling
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5, 0.7, 0.9]),
        
        # Fixed
        'random_state': 42,
        'verbose': 0
    }
    
    # 5-fold CV
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    model = GradientBoostingClassifier(**params)
    
    # Cross-validation
    cv_scores = cross_val_score(
        model, X, y,
        cv=cv,
        scoring='roc_auc',
        n_jobs=-1
    )
    
    cv_mean = cv_scores.mean()
    cv_std = cv_scores.std()
    
    # Penalize high variance across folds (encourages generalization)
    score = cv_mean - 0.5 * cv_std
    
    return score

# Callback to show progress
def callback(study, trial):
    if trial.number % 10 == 0:
        print(f"  Trial {trial.number:3d}: Score={trial.value:.4f} (Best: {study.best_value:.4f})")

# Run optimization
print("\n--- Starting Optuna Optimization ---")
print("  Trials: 100")
print("  CV folds: 5")
print("  Algorithm: Gradient Boosting")
print("  Objective: Maximize (ROC-AUC - 0.5*std)")
print("\nThis may take 2-3 hours...\n")

# Create study
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

# Optimize
study.optimize(
    objective,
    n_trials=100,
    callbacks=[callback],
    show_progress_bar=True,
    n_jobs=1  # GB doesn't parallelize well
)

print("\n" + "="*80)
print("OPTUNA OPTIMIZATION COMPLETE")
print("="*80)

In [ ]:
# =============================================================================
# SHOW BEST RESULTS
# =============================================================================

print("\n--- Best Trial Results ---")
print(f"  Best Score (CV - 0.5*std): {study.best_value:.4f}")

print("\n--- Best Parameters ---")
best_params = study.best_params
for param, value in best_params.items():
    print(f"  {param}: {value}")

# Top 5 trials
print("\n--- Top 5 Trials ---")
best_trials = sorted(study.trials, key=lambda t: t.value, reverse=True)[:5]
for i, trial in enumerate(best_trials, 1):
    print(f"  {i}. Trial {trial.number}: Score={trial.value:.4f}")
    print(f"     n_estimators={trial.params['n_estimators']}, lr={trial.params['learning_rate']:.4f}, depth={trial.params['max_depth']}")

In [ ]:
# =============================================================================
# TRAIN FINAL MODEL WITH BEST PARAMS
# =============================================================================

print("\n" + "="*80)
print("TRAINING FINAL MODEL")
print("="*80)

# Add fixed params
final_params = best_params.copy()
final_params.update({
    'random_state': 42,
    'verbose': 0
})

# Train with best params
final_model = GradientBoostingClassifier(**final_params)

# Get CV score with best params
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(final_model, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f"\nFinal CV Performance:")
print(f"  CV scores: {[f'{s:.4f}' for s in cv_scores]}")
print(f"  Mean: {cv_scores.mean():.4f}")
print(f"  Std: {cv_scores.std():.4f}")

# Train on full data
print("\nTraining on full dataset...")
final_model.fit(X, y)

# Training performance
train_proba = final_model.predict_proba(X)[:, 1]
train_auc = roc_auc_score(y, train_proba)

print(f"\nOverfitting Analysis:")
print(f"  Training AUC: {train_auc:.4f}")
print(f"  CV AUC: {cv_scores.mean():.4f}")
print(f"  Gap: {train_auc - cv_scores.mean():.4f}")

gap = train_auc - cv_scores.mean()
if gap < 0.05:
    print("  ✓ Excellent! Minimal overfitting")
elif gap < 0.10:
    print("  ✓ Good! Acceptable overfitting")
elif gap < 0.15:
    print("  ⚠️ Moderate overfitting")
else:
    print("  ❌ High overfitting")

---
<a id='section12'></a>
## 12. Generate Predictions

Generate predictions for test set and save submission file.

In [ ]:
# =============================================================================
# GENERATE TEST PREDICTIONS
# =============================================================================

print("\n" + "="*80)
print("SECTION 12: GENERATE PREDICTIONS")
print("="*80)

print("\n--- Generating Test Predictions ---")

test_proba = final_model.predict_proba(X_test)[:, 1]

print(f"\nTest prediction statistics:")
print(f"  Min: {test_proba.min():.4f}")
print(f"  Max: {test_proba.max():.4f}")
print(f"  Mean: {test_proba.mean():.4f}")
print(f"  Median: {np.median(test_proba):.4f}")

# Distribution
print(f"\nPrediction distribution:")
bins = [0, 0.05, 0.10, 0.15, 0.20, 0.30, 1.0]
for i in range(len(bins)-1):
    count = ((test_proba >= bins[i]) & (test_proba < bins[i+1])).sum()
    print(f"  {bins[i]:.2f}-{bins[i+1]:.2f}: {count:4d} ({count/len(test_proba)*100:5.1f}%)")

In [ ]:
# =============================================================================
# SAVE SUBMISSION
# =============================================================================

print("\n--- Creating Submission File ---")

# Create submission DataFrame
submission = pd.DataFrame({
    'icustay_id': test_ids,
    'prediction': test_proba
})

# Verify alignment
print(f"  Submission shape: {submission.shape}")
print(f"  First 5 rows:")
print(submission.head())

# Save
submission_file = "dils_corneel_CML_2025.csv"
submission.to_csv(submission_file, index=False)

print(f"\n✓ Saved: {submission_file}")

In [ ]:
# =============================================================================
# FEATURE IMPORTANCE
# =============================================================================

print("\n" + "="*80)
print("TOP 20 FEATURES")
print("="*80)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\n" + feature_importance.head(20).to_string(index=False))

---
## Summary

This notebook presents a complete pipeline for hospital mortality prediction using MIMIC-III ICU data:

**Preprocessing:**
- Hospital history features (previous ICU visits, first visit flag, frequent flyer)
- ICD-9 diagnosis features (n_diagnoses, primary diagnosis, condition flags)
- Age calculation with handling for MIMIC-III de-identification
- Categorical encoding (target encoding for high-cardinality, one-hot for others)
- Medical feature engineering (shock indices, vital sign ranges, severity score)
- Selective feature scaling (only continuous features)

**Model:**
- Gradient Boosting Classifier
- Optuna hyperparameter optimization (100 trials)
- 5-fold stratified cross-validation
- Objective: ROC-AUC - 0.5*std (penalizes variance for better generalization)

**Key Design Decisions:**
1. Temporal ordering preserved before creating history features
2. Target encoding for high-cardinality ICD-9 codes
3. Selective scaling (binary/count features not scaled)
4. Test IDs saved in sorted order for submission alignment